### VectorDatabase

**A vector database is a specialized system designed to store, manage, and query high-dimensional numerical lists called vector embeddings.** 

* **How It Works**
    - **Embeddings**: Machine learning models convert complex data like text, images, and audio into lists of numbers that represent their semantic meaning (context and content). 
    - **Spatial Mapping**: Similar items sit close together in a multi-dimensional mathematical space, while different items sit far apart. 
    - **Similarity Search**: Instead of exact keyword matches, the system uses distance math (like cosine similarity) to find items that mean similar things.
    - **Approximate Nearest Neighbor (ANN):** Algorithms like HNSW quickly scan millions of vectors by trading a tiny bit of accuracy for massive speed gains. 
    
* **Key Benefits**
    - *Unstructured Data*: Easily handles complex files like video, audio, and documents that standard SQL tables struggle with.
    - *Meaning Over Matches*: Understands intent and context rather than just scanning for exact words.
    - *AI Integration:* Powers Retrieval-Augmented Generation (RAG) by letting large language models query external, trusted data

### FAISS

**FAISS (Facebook AI Similarity Search) is an open-source library developed by Meta AI Research for efficient similarity search and clustering of dense vectors.** 

* **What It Does**
    - *Vector Search:* Finds multimedia or text documents with similar meanings by comparing high-dimensional mathematical embeddings.
    - *Scale:* Handles large-scale datasets ranging from millions to billions of vectors.
    - *Speed vs. Accuracy*: Offers exact search options (IndexFlatL2) and approximate nearest neighbor (ANN) indexes to trade a tiny amount of accuracy for massive speed gains

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader('speech.txt')
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=200,
                                      chunk_overlap=30)
docs = text_splitter.split_documents(documents)

docs

/var/folders/0b/_fz3pln11xz8q6vfw486xw4r0000gn/T/ipykernel_73455/2175985715.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/Users/spy/Desktop/agentic-AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Created a chunk of size 465, which is longer than the specified 200
Created a chunk of size 431, which is longer than the specified 200


[Document(metadata={'source': 'speech.txt'}, page_content='Declamation Speech – Bhagat Singh\nBrothers and Sisters of my beloved India,\nToday I stand before you not as a rebel, not as a terrorist as the British call me, but as a son of Mother India who dreams of her freedom. We do not want mercy, we want justice. We do not beg for life, we demand liberty — for every farmer, every worker, every child born on this sacred soil. When I threw the bomb in the Assembly, I did not wish to kill anyone. I wanted the deaf to hear!'),
 Document(metadata={'source': 'speech.txt'}, page_content='It was a cry against injustice, a call for awakening! Our aim was not destruction — it was revolution, Inquilab Zindabad!\nRevolution does not mean bloodshed. Revolution means the birth of a new order — where oppression ends, where exploitation dies, and where freedom breathes in every heart. They may hang me. But remember, they can kill my body, not my ideas. The revolution is not of swords and guns — it is

In [2]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db = FAISS.from_documents(docs, embeddings)
db

/var/folders/0b/_fz3pln11xz8q6vfw486xw4r0000gn/T/ipykernel_73455/2676878469.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [4]:
### Querrying
querry = "What is the role of education?"
docs = db.similarity_search(querry)
docs[0].page_content

'We must educate ourselves, we must awaken our youth, and we must fight — not with hatred, but with courage and conviction.\nLet every Indian rise with pride and say: We will not live as slaves. We will live with dignity. And if we must die, we will die for our Motherland.\nInquilab Zindabad!'

### As a Retriever

**We can also convert the vectorStore into Retriever Class. This alllows us to easily use it in other langchain methods, which largely work with retrievers.**

In [6]:
retriever = db.as_retriever()
docs = retriever.invoke(querry)
docs[0].page_content

'We must educate ourselves, we must awaken our youth, and we must fight — not with hatred, but with courage and conviction.\nLet every Indian rise with pride and say: We will not live as slaves. We will live with dignity. And if we must die, we will die for our Motherland.\nInquilab Zindabad!'

### Similarity Search with Score

**A similarity search with score evaluates how close vector embeddings are in a high-dimensional space, returning matching documents paired with quantitative relevance or distance metrics.**

#### Understanding the Scores

* **Euclidean (L2) Distance (Default)**:
    - Both faiss.IndexFlatL2 and LangChain's default FAISS constructor use L2 distance.
    - Interpretation: The score represents the physical distance between vectors. Lower scores mean higher similarity. A score of 0.0 represents an identical match. 
    
* **Cosine Similarity vs. Distance:**
    - If you need a bounded score between 0 and 1 (where higher means more similar), you must normalize your input vectors to unit length using `faiss.normalize_L2` before adding them to the index and use an Inner Product index (faiss.IndexFlatIP).In LangChain, this can be achieved by utilizing the `_similarity_search_with_relevance_scores `method if your distance strategy is configured for cosine metrics.

In [7]:
## lower the better
docs_and_score = db.similarity_search_with_score(querry)
docs_and_score

[(Document(id='1e6f1e5c-6dd0-435a-a744-a00563c0f617', metadata={'source': 'speech.txt'}, page_content='We must educate ourselves, we must awaken our youth, and we must fight — not with hatred, but with courage and conviction.\nLet every Indian rise with pride and say: We will not live as slaves. We will live with dignity. And if we must die, we will die for our Motherland.\nInquilab Zindabad!'),
  np.float32(467.1219)),
 (Document(id='f4f51d68-2b6b-4e55-90ae-a0ec1e36cbfa', metadata={'source': 'speech.txt'}, page_content='It was a cry against injustice, a call for awakening! Our aim was not destruction — it was revolution, Inquilab Zindabad!\nRevolution does not mean bloodshed. Revolution means the birth of a new order — where oppression ends, where exploitation dies, and where freedom breathes in every heart. They may hang me. But remember, they can kill my body, not my ideas. The revolution is not of swords and guns — it is of minds and souls.'),
  np.float32(482.91354)),
 (Document(i

In [16]:
embedding_vector = embeddings.embed_query(querry)

docs_and_score = db.similarity_search_with_score_by_vector(embedding_vector)
docs_and_score

[(Document(id='1e6f1e5c-6dd0-435a-a744-a00563c0f617', metadata={'source': 'speech.txt'}, page_content='We must educate ourselves, we must awaken our youth, and we must fight — not with hatred, but with courage and conviction.\nLet every Indian rise with pride and say: We will not live as slaves. We will live with dignity. And if we must die, we will die for our Motherland.\nInquilab Zindabad!'),
  np.float32(467.1219)),
 (Document(id='f4f51d68-2b6b-4e55-90ae-a0ec1e36cbfa', metadata={'source': 'speech.txt'}, page_content='It was a cry against injustice, a call for awakening! Our aim was not destruction — it was revolution, Inquilab Zindabad!\nRevolution does not mean bloodshed. Revolution means the birth of a new order — where oppression ends, where exploitation dies, and where freedom breathes in every heart. They may hang me. But remember, they can kill my body, not my ideas. The revolution is not of swords and guns — it is of minds and souls.'),
  np.float32(482.91354)),
 (Document(i

In [17]:
## Saving and Loading
db.save_local("faiss_index")

In [19]:
# loading the db
new_db = FAISS.load_local("faiss_index",
                          embeddings,
                          allow_dangerous_deserialization=True)

docs = new_db.similarity_search(querry)
docs

[Document(id='1e6f1e5c-6dd0-435a-a744-a00563c0f617', metadata={'source': 'speech.txt'}, page_content='We must educate ourselves, we must awaken our youth, and we must fight — not with hatred, but with courage and conviction.\nLet every Indian rise with pride and say: We will not live as slaves. We will live with dignity. And if we must die, we will die for our Motherland.\nInquilab Zindabad!'),
 Document(id='f4f51d68-2b6b-4e55-90ae-a0ec1e36cbfa', metadata={'source': 'speech.txt'}, page_content='It was a cry against injustice, a call for awakening! Our aim was not destruction — it was revolution, Inquilab Zindabad!\nRevolution does not mean bloodshed. Revolution means the birth of a new order — where oppression ends, where exploitation dies, and where freedom breathes in every heart. They may hang me. But remember, they can kill my body, not my ideas. The revolution is not of swords and guns — it is of minds and souls.'),
 Document(id='95200590-487d-4614-9bec-9f9ea52d5b15', metadata={'s

### Chroma

**Chroma is a AI-native open source vector database which is Fast, serverless, and scalable infrastructure supporting vector, full-text, regex, and metadata search. Built on object storage and trusted by millions of developers. Open-source Apache 2.0.**

In [21]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [22]:
loader = TextLoader('speech.txt')
data = loader.load()
data

[Document(metadata={'source': 'speech.txt'}, page_content='Declamation Speech – Bhagat Singh\nBrothers and Sisters of my beloved India,\nToday I stand before you not as a rebel, not as a terrorist as the British call me, but as a son of Mother India who dreams of her freedom. We do not want mercy, we want justice. We do not beg for life, we demand liberty — for every farmer, every worker, every child born on this sacred soil. When I threw the bomb in the Assembly, I did not wish to kill anyone. I wanted the deaf to hear!\n\n\nIt was a cry against injustice, a call for awakening! Our aim was not destruction — it was revolution, Inquilab Zindabad!\nRevolution does not mean bloodshed. Revolution means the birth of a new order — where oppression ends, where exploitation dies, and where freedom breathes in every heart. They may hang me. But remember, they can kill my body, not my ideas. The revolution is not of swords and guns — it is of minds and souls.\n\n\nWe must educate ourselves, we m

In [23]:
## split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200,
                                               chunk_overlap=20)
splits = text_splitter.split_documents(data)


In [24]:
## vectorstore and embeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectordb = Chroma.from_documents(documents=splits,
                                 embedding=embeddings)

vectordb

In [25]:
### Querrying
querry = "What is the role of education?"
docs = vectordb.similarity_search(querry)
docs[0].page_content

'We must educate ourselves, we must awaken our youth, and we must fight — not with hatred, but with courage and conviction.'

In [26]:
## save and load
vectordb = Chroma.from_documents(documents=splits,
                                 embedding=embeddings,
                                 persist_directory="./chroma_db")

In [27]:
## load chroma db
chroma_db = Chroma(persist_directory="./chroma_db",
                   embedding_function=embeddings)

docs = chroma_db.similarity_search(querry)
docs[0].page_content

'We must educate ourselves, we must awaken our youth, and we must fight — not with hatred, but with courage and conviction.'

In [28]:
chroma_db.similarity_search_with_score(querry)

[(Document(id='00f51df7-9d35-49ec-855c-6465118b73e9', metadata={'source': 'speech.txt'}, page_content='We must educate ourselves, we must awaken our youth, and we must fight — not with hatred, but with courage and conviction.'),
  436.59820556640625),
 (Document(id='24591d70-cda8-49bf-99c5-8c9859db1390', metadata={'source': 'speech.txt'}, page_content='We do not beg for life, we demand liberty — for every farmer, every worker, every child born on this sacred soil. When I threw the bomb in the Assembly, I did not wish to kill anyone. I wanted the'),
  497.2198486328125),
 (Document(id='6e4a9d96-ce7c-4049-a913-0d8921e66504', metadata={'source': 'speech.txt'}, page_content='Revolution does not mean bloodshed. Revolution means the birth of a new order — where oppression ends, where exploitation dies, and where freedom breathes in every heart. They may hang me. But'),
  498.20953369140625),
 (Document(id='fe8596d2-4d5d-48b9-be7d-e4a55fdd08c9', metadata={'source': 'speech.txt'}, page_content

In [29]:
### Retriever
retriever = chroma_db.as_retriever()
retriever.invoke(querry)[0].page_content

'We must educate ourselves, we must awaken our youth, and we must fight — not with hatred, but with courage and conviction.'